# SigLIP2-SO400M Macro Router — ONNX Conversion
Export .pt → .onnx for TensorRT deployment

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

### ONNX Export

In [ ]:
"""
============================================================
   ONNX EXPORT — SigLIP2_SO400M (FIXED)
   Converts .pt checkpoint → .onnx for TensorRT deployment
============================================================
Usage (Colab):
    !pip install --upgrade transformers>=4.56.0 onnx onnxruntime-gpu --quiet
    Then run this cell.

FIXES vs previous version:
  1. Use SAME dummy_input (batch=1) for PT test, ONNX export, and validation
  2. Removed dynamic_axes (TRT uses fixed batch=1, matches DINOv3 export)
  3. Proper PT vs ONNX comparison on identical input
"""

import os
import torch
import torch.nn as nn
import numpy as np
import time

# !pip install onnx onnxruntime

# ==============================================================
# CONFIG
# ==============================================================

CHECKPOINT_PATH = "/content/drive/Shareddrives/Garment Type/Complete_dataset/model_comparison_18th/SigLIP2_SO400M_best.pt"
ONNX_OUTPUT = "/content/drive/Shareddrives/Garment Type/Complete_dataset/model_comparison_18th/SigLIP2_SO400M.onnx"

MODEL_ID = "google/siglip2-so400m-patch14-384"
IMG_SIZE = 384
NUM_CLASSES = 4
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
OPSET_VERSION = 17

# ==============================================================
# WRAPPER MODEL
# ==============================================================

class ClassifierHead(nn.Module):
    def __init__(self, hidden_size, num_classes, dropout_rate=0.3):
        super().__init__()
        self.dropout = nn.Dropout(dropout_rate)
        self.fc = nn.Linear(hidden_size, num_classes)
    def forward(self, x):
        return self.fc(self.dropout(x))


class SigLIP2ForExport(nn.Module):
    """
    Single module: SigLIP2 vision backbone → pooler → classifier → softmax.
    Wraps the HuggingFace model so ONNX export sees a clean forward(pixel_values) → probs.
    """
    def __init__(self, vision_model, classifier):
        super().__init__()
        self.vision_model = vision_model
        self.classifier = classifier

    def forward(self, pixel_values):
        outputs = self.vision_model(pixel_values=pixel_values)
        features = outputs.pooler_output
        logits = self.classifier(features)
        probs = torch.softmax(logits, dim=1)
        return probs


# ==============================================================
# MAIN
# ==============================================================

def main():
    print("=" * 60)
    print("  ONNX EXPORT — SigLIP2_SO400M (FIXED)")
    print("=" * 60)

    # 1. Load checkpoint
    print(f"\nLoading checkpoint: {CHECKPOINT_PATH}")
    ckpt = torch.load(CHECKPOINT_PATH, map_location=DEVICE, weights_only=False)
    hidden_size = ckpt["hidden_size"]
    print(f"  F1: {ckpt['best_f1']:.4f}, Epoch: {ckpt['epoch']}, Hidden: {hidden_size}")

    # 2. Build model architecture
    print(f"\nLoading SigLIP2 vision backbone...")
    try:
        from transformers import Siglip2VisionModel
        vision_model = Siglip2VisionModel.from_pretrained(MODEL_ID)
        print(f"  Loaded via Siglip2VisionModel")
    except Exception:
        try:
            from transformers import SiglipVisionModel
            vision_model = SiglipVisionModel.from_pretrained(MODEL_ID)
            print(f"  Loaded via SiglipVisionModel")
        except Exception:
            from transformers import AutoModel
            full = AutoModel.from_pretrained(MODEL_ID)
            vision_model = full.vision_model
            del full
            print(f"  Loaded via AutoModel fallback")

    # Load trained weights
    vision_model.load_state_dict(ckpt["vision_model"])

    classifier = ClassifierHead(hidden_size, NUM_CLASSES)
    classifier.load_state_dict(ckpt["classifier"])

    export_model = SigLIP2ForExport(vision_model, classifier)
    export_model = export_model.to(DEVICE)
    export_model.eval()

    total_params = sum(p.numel() for p in export_model.parameters())
    print(f"  Total params: {total_params:,}")

    # 3. Create ONE dummy input — reuse for PT test, ONNX export, and validation
    dummy_input = torch.randn(1, 3, IMG_SIZE, IMG_SIZE, device=DEVICE)

    # 4. Test PyTorch inference
    print(f"\nTesting PyTorch inference...")
    with torch.no_grad():
        pt_output = export_model(dummy_input)
    print(f"  Output shape: {pt_output.shape}")
    print(f"  Sum of probs: {pt_output.sum().item():.4f} (should be 1.0)")
    print(f"  Probs: {pt_output.cpu().numpy().round(4)}")

    # 5. Export to ONNX (STATIC batch=1, same as DINOv3 export)
    print(f"\nExporting to ONNX (STATIC batch=1, opset={OPSET_VERSION})...")
    print(f"  (SigLIP2 is 428M params — this may take 1-2 minutes)")
    os.makedirs(os.path.dirname(ONNX_OUTPUT), exist_ok=True)

    torch.onnx.export(
        export_model,
        dummy_input,                    # batch=1 — SAME input used for PT test
        ONNX_OUTPUT,
        opset_version=OPSET_VERSION,
        input_names=["pixel_values"],
        output_names=["probabilities"],
        # No dynamic_axes — TRT uses fixed batch=1 (matches DINOv3 export)
        training=torch.onnx.TrainingMode.EVAL,
        do_constant_folding=True,
        dynamo=False,
        keep_initializers_as_inputs=False,
    )

    onnx_size_mb = os.path.getsize(ONNX_OUTPUT) / (1024 * 1024)
    print(f"  ✅ Saved: {ONNX_OUTPUT}")
    print(f"  File size: {onnx_size_mb:.1f} MB")

    # 6. Validate ONNX model
    print(f"\nValidating ONNX model...")
    import onnx
    onnx_model = onnx.load(ONNX_OUTPUT)
    onnx.checker.check_model(onnx_model)
    print(f"  ✅ ONNX model is valid")

    # 7. Test ONNX Runtime inference — SAME input as PyTorch
    print(f"\nTesting ONNX Runtime inference...")
    import onnxruntime as ort

    providers = ['CUDAExecutionProvider', 'CPUExecutionProvider']
    session = ort.InferenceSession(ONNX_OUTPUT, providers=providers)
    actual_provider = session.get_providers()[0]
    print(f"  Provider: {actual_provider}")

    input_np = dummy_input.cpu().numpy()                          # SAME input
    ort_output = session.run(None, {"pixel_values": input_np})[0]

    print(f"  ONNX output shape: {ort_output.shape}")
    print(f"  ONNX probs: {ort_output.round(4)}")

    # 8. Compare PyTorch vs ONNX — now comparing SAME input → SAME expected output
    pt_np = pt_output.cpu().numpy()
    max_diff = np.abs(pt_np - ort_output).max()
    print(f"\n  Max difference (PT vs ONNX): {max_diff:.8f}")
    assert max_diff < 1e-3, f"MISMATCH! max_diff={max_diff}"
    print(f"  ✅ PyTorch and ONNX outputs match (diff < 1e-3)")

    # 9. Benchmark ONNX speed
    print(f"\nBenchmarking ONNX speed (50 runs)...")
    for _ in range(5):
        session.run(None, {"pixel_values": input_np})
    times = []
    for _ in range(50):
        t0 = time.perf_counter()
        session.run(None, {"pixel_values": input_np})
        times.append(time.perf_counter() - t0)
    avg_ms = np.mean(times) * 1000
    fps = 1000 / avg_ms
    print(f"  Avg: {avg_ms:.1f} ms/image ({fps:.1f} FPS)")

    print(f"\n{'='*60}")
    print(f"  DONE — Ready for TensorRT conversion on Orin")
    print(f"  trtexec --onnx=SigLIP2_SO400M.onnx --saveEngine=SigLIP2_SO400M_fp16.engine --fp16 --memPoolSize=workspace:8192M")
    print(f"{'='*60}")


if __name__ == "__main__":
    main()

### ONNX Inference Verification

In [ ]:
"""
============================================================
   ONNX INFERENCE VERIFICATION — Both Models
   Tests DINOv3 and SigLIP2 .onnx on real garment images
============================================================
Usage (Colab):
    !pip install onnxruntime-gpu --quiet
    Then run this cell.
"""

import os
import cv2
import json
import numpy as np
from PIL import Image
import time
import onnxruntime as ort

# ==============================================================
# CONFIG
# ==============================================================

DINOV3_ONNX = "/content/drive/Shareddrives/Garment Type/Complete_dataset/model_comparison_18th/DINOv3_ViT_Base.onnx"
SIGLIP2_ONNX = "/content/drive/Shareddrives/Garment Type/Complete_dataset/model_comparison_18th/SigLIP2_SO400M.onnx"

# Production configs (for thresholds)
DINOV3_CONFIG = "/content/drive/Shareddrives/Garment Type/Complete_dataset/model_comparison_18th/dinov3_production_config.json"
SIGLIP2_CONFIG = "/content/drive/Shareddrives/Garment Type/Complete_dataset/model_comparison_18th/siglip2_production_config.json"

# Test images — pick a folder with known labels
TEST_DIR = "/content/drive/Shareddrives/Garment Type/classified_images_13_14_test"
MAX_TEST_IMAGES = 1300  # Test on first N images for quick verification

IMG_SIZE = 384

CLASS_NAMES = ["Upper", "Lower", "Underwear", "Other"]

# Normalisation per model
DINOV3_MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32).reshape(3, 1, 1)
DINOV3_STD = np.array([0.229, 0.224, 0.225], dtype=np.float32).reshape(3, 1, 1)

SIGLIP2_MEAN = np.array([0.5, 0.5, 0.5], dtype=np.float32).reshape(3, 1, 1)
SIGLIP2_STD = np.array([0.5, 0.5, 0.5], dtype=np.float32).reshape(3, 1, 1)

# ==============================================================
# PREPROCESSING (exact match to training)
# ==============================================================

BACKGROUND_COLOR = (128, 128, 128)
MARGIN_RATIO = 0.08
BLACK_THRESH = 3

FOLDER_TO_MACRO = {
    "blazer": "Upper", "jumpers": "Upper", "shirt": "Upper",
    "t-shirt": "Upper", "dresses": "Upper", "fleece": "Upper",
    "jeans": "Lower", "trousers": "Lower", "jogging_bottoms": "Lower",
    "skirts": "Lower", "bra": "Underwear", "kincker": "Underwear",
    "knicker": "Underwear", "towel": "Other",
    "Blazer": "Upper", "Jumpers": "Upper", "Shirt": "Upper",
    "T-shirt": "Upper", "Dresses": "Upper", "Fleece": "Upper",
    "Jeans": "Lower", "Trousers": "Lower", "Jogging_Bottoms": "Lower",
    "Skirts": "Lower", "Bra": "Underwear", "Knicker": "Underwear",
    "Towel": "Other",
}

CLASS_TO_IDX = {"Upper": 0, "Lower": 1, "Underwear": 2, "Other": 3}
SKIP_FOLDERS = {"13_14th_test", "unknown", "to_test", "SKIP"}


def preprocess_crop(img_np):
    h, w = img_np.shape[:2]
    max_channel = np.max(img_np, axis=2)
    mask = np.where(max_channel > BLACK_THRESH, 255, 0).astype(np.uint8)
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (7, 7))
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel, iterations=3)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel, iterations=1)
    flood_mask = np.zeros((h + 2, w + 2), dtype=np.uint8)
    inverted = cv2.bitwise_not(mask)
    for (sx, sy) in [(0, 0), (w-1, 0), (0, h-1), (w-1, h-1)]:
        if inverted[sy, sx] == 255:
            cv2.floodFill(inverted, flood_mask, (sx, sy), 0)
    mask = cv2.bitwise_or(mask, inverted)
    gray_bg = np.full_like(img_np, BACKGROUND_COLOR, dtype=np.uint8)
    mask_3ch = cv2.merge([mask, mask, mask])
    new_crop = np.where(mask_3ch > 0, img_np, gray_bg)
    x, y, bw, bh = cv2.boundingRect(mask)
    if bw == 0 or bh == 0:
        side = max(h, w)
        square = np.full((side, side, 3), BACKGROUND_COLOR, dtype=np.uint8)
        return square
    mx, my = int(bw * MARGIN_RATIO), int(bh * MARGIN_RATIO)
    x1, y1 = max(0, x - mx), max(0, y - my)
    x2, y2 = min(w, x + bw + mx), min(h, y + bh + my)
    cropped = new_crop[y1:y2, x1:x2]
    ch, cw = cropped.shape[:2]
    side = max(ch, cw)
    square = np.full((side, side, 3), BACKGROUND_COLOR, dtype=np.uint8)
    oy, ox = (side - ch) // 2, (side - cw) // 2
    square[oy:oy+ch, ox:ox+cw] = cropped
    return square


def prepare_input(img_path, mean, std):
    """Full pipeline: read → preprocess → resize → normalize → NCHW float32."""
    img_bgr = cv2.imread(img_path)
    if img_bgr is None:
        return None

    # Preprocess (gray bg, pad-to-square)
    square_bgr = preprocess_crop(img_bgr)

    # Resize
    resized = cv2.resize(square_bgr, (IMG_SIZE, IMG_SIZE))

    # BGR → RGB → float [0,1] → CHW
    rgb = cv2.cvtColor(resized, cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0
    chw = np.transpose(rgb, (2, 0, 1))  # HWC → CHW

    # Normalize
    chw = (chw - mean) / std

    # Add batch dim
    return chw[np.newaxis, ...]  # (1, 3, 384, 384)


# ==============================================================
# COLLECT TEST SAMPLES
# ==============================================================

def collect_test_samples(root, max_samples=200):
    samples = []
    for folder_name in sorted(os.listdir(root)):
        folder_path = os.path.join(root, folder_name)
        if not os.path.isdir(folder_path):
            continue
        if folder_name in SKIP_FOLDERS:
            continue
        macro = FOLDER_TO_MACRO.get(folder_name)
        if macro is None:
            continue
        label = CLASS_TO_IDX[macro]
        for f in sorted(os.listdir(folder_path)):
            if f.lower().endswith((".jpg", ".jpeg", ".png")):
                samples.append((os.path.join(folder_path, f), label, folder_name))
                if len(samples) >= max_samples:
                    return samples
    return samples


# ==============================================================
# RUN INFERENCE
# ==============================================================

def run_onnx_eval(session, samples, mean, std, model_name, thresholds=None):
    """Run ONNX model on samples and print results."""
    correct = 0
    total = 0
    errors = []
    accepted = 0
    accepted_correct = 0
    rejected = 0
    times = []

    for path, label, folder in samples:
        input_np = prepare_input(path, mean, std)
        if input_np is None:
            continue

        t0 = time.perf_counter()
        probs = session.run(None, {"pixel_values": input_np})[0][0]
        times.append(time.perf_counter() - t0)

        pred = int(np.argmax(probs))
        conf = float(probs[pred])
        total += 1

        if pred == label:
            correct += 1

        # Threshold check
        if thresholds:
            thresh = thresholds.get(CLASS_NAMES[pred], 0.5)
            if conf >= thresh:
                accepted += 1
                if pred == label:
                    accepted_correct += 1
            else:
                rejected += 1

        if pred != label:
            errors.append({
                "file": os.path.basename(path),
                "folder": folder,
                "true": CLASS_NAMES[label],
                "pred": CLASS_NAMES[pred],
                "conf": conf,
            })

    accuracy = correct / total if total > 0 else 0
    avg_ms = np.mean(times) * 1000
    fps = 1000 / avg_ms

    print(f"\n  📊 {model_name} — ONNX Inference Results")
    print(f"  {'─'*50}")
    print(f"  Accuracy:    {accuracy:.4f} ({correct}/{total})")
    print(f"  Errors:      {len(errors)}")
    print(f"  Speed:       {avg_ms:.1f} ms/img ({fps:.1f} FPS)")

    if thresholds:
        acc_prec = accepted_correct / accepted if accepted > 0 else 0
        print(f"\n  Threshold rejection:")
        print(f"    Accepted:  {accepted} ({accepted/total*100:.1f}%)")
        print(f"    Rejected:  {rejected} ({rejected/total*100:.1f}%) → 'Unknown'")
        print(f"    Accepted precision: {acc_prec:.4f}")

    if errors:
        print(f"\n  Top 10 mistakes (sorted by confidence):")
        errors.sort(key=lambda x: x["conf"], reverse=True)
        for e in errors[:10]:
            print(f"    {e['file']:40s} | {e['folder']:15s} "
                  f"true={e['true']:10s} pred={e['pred']:10s} conf={e['conf']:.3f}")

    return accuracy, fps


# ==============================================================
# MAIN
# ==============================================================

def main():
    print("=" * 60)
    print("  ONNX INFERENCE VERIFICATION — Both Models")
    print("=" * 60)

    # Collect test samples
    print(f"\nCollecting test samples from: {TEST_DIR}")
    samples = collect_test_samples(TEST_DIR, MAX_TEST_IMAGES)
    print(f"  Loaded {len(samples)} samples")

    if len(samples) == 0:
        print("❌ No samples found!")
        return

    results = {}

    # ── DINOv3 ──
    if os.path.exists(DINOV3_ONNX):
        print(f"\n{'='*60}")
        print(f"  Loading DINOv3 ONNX: {DINOV3_ONNX}")
        size_mb = os.path.getsize(DINOV3_ONNX) / (1024 * 1024)
        print(f"  Size: {size_mb:.1f} MB")

        providers = ['CUDAExecutionProvider', 'CPUExecutionProvider']
        session = ort.InferenceSession(DINOV3_ONNX, providers=providers)
        print(f"  Provider: {session.get_providers()[0]}")

        # Load thresholds
        dinov3_thresh = None
        if os.path.exists(DINOV3_CONFIG):
            with open(DINOV3_CONFIG) as f:
                dinov3_thresh = json.load(f).get("class_thresholds")
            print(f"  Thresholds: {dinov3_thresh}")

        acc, fps = run_onnx_eval(session, samples, DINOV3_MEAN, DINOV3_STD,
                                  "DINOv3_ViT_Base", dinov3_thresh)
        results["DINOv3"] = {"accuracy": acc, "fps": fps}
        del session
    else:
        print(f"\n⚠️  DINOv3 ONNX not found: {DINOV3_ONNX}")

    # ── SigLIP2 ──
    if os.path.exists(SIGLIP2_ONNX):
        print(f"\n{'='*60}")
        print(f"  Loading SigLIP2 ONNX: {SIGLIP2_ONNX}")
        size_mb = os.path.getsize(SIGLIP2_ONNX) / (1024 * 1024)
        print(f"  Size: {size_mb:.1f} MB")

        providers = ['CUDAExecutionProvider', 'CPUExecutionProvider']
        session = ort.InferenceSession(SIGLIP2_ONNX, providers=providers)
        print(f"  Provider: {session.get_providers()[0]}")

        # Load thresholds
        siglip2_thresh = None
        if os.path.exists(SIGLIP2_CONFIG):
            with open(SIGLIP2_CONFIG) as f:
                siglip2_thresh = json.load(f).get("class_thresholds")
            print(f"  Thresholds: {siglip2_thresh}")

        acc, fps = run_onnx_eval(session, samples, SIGLIP2_MEAN, SIGLIP2_STD,
                                  "SigLIP2_SO400M", siglip2_thresh)
        results["SigLIP2"] = {"accuracy": acc, "fps": fps}
        del session
    else:
        print(f"\n⚠️  SigLIP2 ONNX not found: {SIGLIP2_ONNX}")

    # ── Comparison ──
    if len(results) == 2:
        print(f"\n\n{'='*60}")
        print(f"  SIDE-BY-SIDE COMPARISON (ONNX)")
        print(f"{'='*60}")
        print(f"  {'Model':20s} {'Accuracy':>10s} {'FPS':>10s} {'Size':>10s}")
        print(f"  {'─'*50}")
        for name, r in results.items():
            onnx_path = DINOV3_ONNX if name == "DINOv3" else SIGLIP2_ONNX
            size = os.path.getsize(onnx_path) / (1024 * 1024)
            print(f"  {name:20s} {r['accuracy']:>10.4f} {r['fps']:>9.1f}x {size:>9.1f}MB")


if __name__ == "__main__":
    main()